# Notebook 04 — Agent Graph (โครงสร้างจริงของโปรเจค)

## Main Graph Overview

```
                         START
                           │
                    ┌──────▼──────┐
                    │ session_gate│  เตรียม session
                    └──────┬──────┘
                           │ (conditional)
              ┌────────────┼────────────┐
              │            │            │
      mode=HANDOFF    status=ACTIVE  otherwise
              │            │            │
     ┌────────▼────┐ ┌─────▼──────┐ ┌──▼──────────────┐
     │handoff_mode │ │resume_router│ │ intent_classify  │
     └─────────────┘ └─────┬──────┘ └──────────┬────────┘
              │            │                   │
             END      (conditional)       (conditional)
                           │                   │
                    ┌──────┘     ┌─────────────┘
                    │            │
           ┌────────▼────────────▼────────┐
           │  route_module determines     │
           │  which module node to run    │
           └───┬──────┬──────┬────────┬───┘
               │      │      │        │
       hospital_info nurse package concierge
           (subgraph) (subgraph) ...
               │      │      │        │
               └──────┴──────┴────────┘
                              │
                             END
```


## Node 1: session_gate

**ไฟล์:** `src/agent/stub_nodes/session_gate_stub.py`

Node แรกของทุก request — เตรียม context ก่อนประมวลผล

In [ ]:
# จำลอง session_gate_stub.py

async def node_session_gate_prepare(state: dict) -> dict:
    """
    Resolves last_user_message จาก:
    1. latest_user_input (ข้อความใหม่ที่เพิ่งส่งมา)
    2. หรือ messages history (fallback)
    """
    # ดึงจาก latest_user_input ก่อน
    last_user_message = state.get("latest_user_input", "")
    
    # ถ้าไม่มี ดึงจาก messages
    if not last_user_message:
        for msg in reversed(state.get("messages", [])):
            if getattr(msg, 'type', None) == 'human':
                last_user_message = msg.content
                break
    
    return {"last_user_message": last_user_message}


def router_after_session_gate(state: dict) -> str:
    """
    Routing logic หลัง session_gate:
    
    1. HANDOFF mode → ไปจัดการ handoff
    2. Task ACTIVE → ไป resume_router (ทำ task เดิมต่อ)
    3. อื่นๆ → intent_classify (task ใหม่)
    """
    mode = state.get("conversation_mode", "AI")
    status = state.get("task_status", "COMPLETED")
    
    if mode == "HANDOFF":
        return "handoff_mode"
    if status == "ACTIVE":
        return "resume_router"
    return "intent_classify"


# ทดสอบ
import asyncio

test_cases = [
    {"name": "New task",    "state": {"conversation_mode": "AI",      "task_status": "COMPLETED", "latest_user_input": "สวัสดี"}},
    {"name": "Active task", "state": {"conversation_mode": "AI",      "task_status": "ACTIVE",    "latest_user_input": "ต่อ"}},
    {"name": "Handoff",     "state": {"conversation_mode": "HANDOFF", "task_status": "COMPLETED", "latest_user_input": "hello"}},
]

for tc in test_cases:
    update = asyncio.run(node_session_gate_prepare(tc['state']))
    route = router_after_session_gate(tc['state'])
    print(f"[{tc['name']}]")
    print(f"  last_user_message = {update['last_user_message']!r}")
    print(f"  route → {route}")
    print()

## Node 2: intent_classify

**ไฟล์:** `src/agent/stub_nodes/intent_router/node.py`

ใช้ LLM 2 ครั้ง:
1. สรุป **user_goal** (ผู้ใช้ต้องการอะไร)
2. ตัดสิน **route_module** (ควรส่งไป module ไหน)

In [ ]:
# จำลอง intent_classify_stub()

# LLM Structured Output Schemas
from typing import TypedDict, Literal

class UserGoalOut(TypedDict):
    """Schema สำหรับ user_goal extraction"""
    user_goal: str  # สรุปเป้าหมายของผู้ใช้เป็น 1 ประโยค

class RouteOut(TypedDict):
    """Schema สำหรับ routing decision"""
    route_module: Literal[
        "hospital_info_agent_module",
        "nurse_agent_module",
        "package_agent_module",
        "concierge_agent_module"
    ]


async def intent_classify_node_detail(state: dict) -> dict:
    """
    สำคัญ! Node นี้ทำงาน 2 ขั้นตอน:
    
    1. Postback Fast-track:
       - ถ้า postback_data มีข้อมูล → skip LLM
       - route ตรงไปยัง module ที่ระบุ
    
    2. LLM Classification (กรณีข้อความธรรมดา):
       a) _call_user_goal() → สรุป user goal
       b) _call_route() → ตัดสิน module
    """
    postback_data = state.get("postback_data", "")
    user_msg = state.get("last_user_message", "")
    
    # ─── Postback Fast-track ───
    if postback_data:
        # Parse postback data: "action=concierge_menu&..." 
        params = dict(p.split("=") for p in postback_data.split("&") if "=" in p)
        action = params.get("action", "")
        
        # ถ้า action=concierge_menu → ตอบทันทีโดยไม่ผ่าน module
        if action == "concierge_menu":
            print(f"[intent_classify] Postback fast-track: {action}")
            return {
                "route_module": "concierge_agent_module",
                "task_status": "ACTIVE",
                "user_goal": "เปิดเมนู Concierge",
            }
        
        # Postbacks อื่น → route ตาม action
        module_map = {
            "hospital_agent": "hospital_info_agent_module",
            "nurse_agent":    "nurse_agent_module",
        }
        if action in module_map:
            return {
                "route_module": module_map[action],
                "task_status": "ACTIVE",
                "user_goal": f"postback: {action}",
            }
    
    # ─── LLM Classification ───
    print(f"[intent_classify] Classifying: '{user_msg}'")
    
    # Step 1: สรุป user goal
    # llm_with_schema = make_llm(schema=UserGoalOut)
    # goal_result = await llm_with_schema.ainvoke([HumanMessage(user_msg)])
    # user_goal = goal_result['user_goal']
    user_goal = f"ต้องการ: {user_msg}"  # mock
    print(f"[intent_classify] user_goal: {user_goal!r}")
    
    # Step 2: Route ไป module
    # llm_with_route = make_llm(schema=RouteOut)
    # route_result = await llm_with_route.ainvoke([HumanMessage(user_goal)])
    # route = route_result['route_module']
    
    # Mock routing logic
    keywords_hospital = ["โรงพยาบาล", "ที่จอด", "เวลาเปิด", "ประกัน"]
    route = "concierge_agent_module"  # default
    if any(kw in user_msg for kw in keywords_hospital):
        route = "hospital_info_agent_module"
    print(f"[intent_classify] route: {route!r}")
    
    return {
        "user_goal": user_goal,
        "route_module": route,
        "task_status": "ACTIVE",
    }


# ทดสอบ
test_intents = [
    {"last_user_message": "ที่จอดรถโรงพยาบาล A อยู่ที่ไหน", "postback_data": ""},
    {"last_user_message": "", "postback_data": "action=concierge_menu"},
    {"last_user_message": "สวัสดีครับ", "postback_data": ""},
]

print("=== Intent Classification Tests ===")
for state in test_intents:
    result = asyncio.run(intent_classify_node_detail(state))
    print(f"  Input: {state['last_user_message']!r} / postback: {state['postback_data']!r}")
    print(f"  → route: {result['route_module']}")
    print()

## Node 3: Hospital Info Module (Subgraph)

**ไฟล์:** `src/agent/stub_nodes/hospital_info_agent_module/subgraph.py`

Module นี้มีขั้นตอนการทำงานหลายขั้น ถูก implement เป็น subgraph:

In [ ]:
# Hospital Info Module Flow

print("Hospital Info Module Subgraph:")
print()
print("""
START
  ↓
route_hospital_info  ← Router: ตัดสินใจว่าจะทำอะไรต่อ
  │
  ├─► extract_info_tool      ← สกัด topic + hospital จาก user_goal
  │      ↓
  │   route_hospital_info (loop กลับมา)
  │
  ├─► ask_more_info_tool     ← ถามผู้ใช้ว่าต้องการข้อมูลอะไร/โรงพยาบาลไหน
  │      ↓ END
  │
  ├─► fetch_data_tool        ← ดึงข้อมูลจาก CSV index
  │      ↓
  │   route_hospital_info (loop กลับมา)
  │  
  ├─► oos_check_tool         ← ตรวจสอบว่า topic อยู่นอก scope ไหม
  │      ↓ END
  │
  ├─► build_answer_tool      ← สร้างคำตอบสุดท้าย
  │      ↓ END
  │
  └─► handoff_postback_tool  ← เตรียม handoff ไปที่ human agent
         ↓ END
""")

# Router Logic
def router_hospital_info(state: dict) -> str:
    """
    เลือก tool ถัดไปใน hospital info module
    ดูจาก state flags
    """
    # ถ้ายังไม่ได้สกัด topics → extract ก่อน
    if not state.get("topic_intents") and not state.get("need_ask_more_info"):
        return "extract_info_tool"
    
    # ถ้าต้องถามข้อมูลเพิ่ม
    if state.get("need_ask_more_info"):
        return "ask_more_info_tool"
    
    # ถ้าต้องตรวจ out-of-scope
    if state.get("need_oos_check"):
        return "oos_check_tool"
    
    # ถ้ามี topics แต่ยังไม่มีข้อมูล → fetch จาก CSV
    if state.get("need_retrieval") and not state.get("hospital_results"):
        return "fetch_data_tool"
    
    # ถ้าต้อง handoff
    if state.get("use_handoff"):
        return "handoff_postback_tool"
    
    # พร้อมตอบ
    if state.get("ready_to_answer"):
        return "build_answer_tool"
    
    return "build_answer_tool"  # default


# ทดสอบ routing ใน module
test_module_states = [
    {"name": "Initial",           "topic_intents": [],   "need_ask_more_info": False, "ready_to_answer": False},
    {"name": "After extract",     "topic_intents": ["ที่จอดรถ"], "need_retrieval": True, "hospital_results": []},
    {"name": "After fetch",       "topic_intents": ["ที่จอดรถ"], "hospital_results": [{"data": "..."}], "ready_to_answer": True},
    {"name": "Need more info",    "topic_intents": [],   "need_ask_more_info": True},
]

print("\nRouter decisions:")
for state in test_module_states:
    route = router_hospital_info(state)
    print(f"  [{state['name']}] → {route}")

## Node 4: Handoff Mode

**ไฟล์:** `src/agent/stub_nodes/handoff_mode_stub.py`

เมื่อ conversation_mode = "HANDOFF" → bot หยุดตอบ ส่งต่อไปที่ human agent

In [ ]:
# จำลอง handoff_mode_stub()

async def handoff_mode_node(state: dict) -> dict:
    """
    Handoff Mode:
    - ถ้า postback action=ai_mode → ออกจาก handoff กลับมา AI
    - ถ้าเป็นข้อความธรรมดา → block (ไม่ตอบ เพราะ human agent ดูแลอยู่)
    """
    postback_data = state.get("postback_data", "")
    
    if postback_data:
        params = dict(p.split("=") for p in postback_data.split("&") if "=" in p)
        action = params.get("action", "")
        
        if action == "ai_mode":
            # ผู้ใช้กดปุ่ม "กลับสู่ AI" บน Rich Menu
            print("[handoff_mode] User returned to AI mode")
            
            # Unlink Rich Menu
            # await unlink_rich_menu(user_id, token)
            
            return {
                "conversation_mode": "AI",
                "final_answer": "ยินดีต้อนรับกลับสู่ AI Assistant ครับ! มีอะไรให้ช่วยไหม?",
            }
    
    # ข้อความธรรมดาใน HANDOFF mode → block
    print("[handoff_mode] Message blocked (in HANDOFF mode)")
    return {
        "final_answer": None,  # ไม่ตอบ
        "line_messages": [],
    }


# ทดสอบ
test_handoff_cases = [
    {"desc": "User in HANDOFF sends text",  "postback_data": "",              "last_user_message": "hello"},
    {"desc": "User presses 'Return to AI'", "postback_data": "action=ai_mode", "last_user_message": ""},
]

for case in test_handoff_cases:
    print(f"Test: {case['desc']}")
    result = asyncio.run(handoff_mode_node(case))
    print(f"  final_answer: {result.get('final_answer')!r}")
    print(f"  conversation_mode: {result.get('conversation_mode', 'unchanged')!r}")
    print()

## LLM Factory — `src/agent/shared/llm_factory.py`

โปรเจคใช้ factory pattern เพื่อสร้าง LLM instances

In [ ]:
# จำลอง llm_factory.py
# โค้ดจริง: src/agent/shared/llm_factory.py

from functools import lru_cache
from typing import Any, Optional, Type

@lru_cache(maxsize=None)
def make_llm(
    provider: str = "openai",
    model: str = "gpt-4o-mini",
    temperature: float = 0,
    schema: Optional[Type] = None,
):
    """
    สร้าง LLM instance พร้อม cache (ไม่สร้างใหม่ซ้ำๆ)
    
    schema: ถ้าระบุ → ใช้ structured output (return dict ตาม schema)
    """
    # from langchain_openai import ChatOpenAI
    # llm = ChatOpenAI(model=model, temperature=temperature)
    # if schema:
    #     return llm.with_structured_output(schema)
    # return llm
    
    print(f"  Creating LLM: provider={provider}, model={model}, temp={temperature}")
    if schema:
        print(f"  With structured output: {schema.__name__ if hasattr(schema, '__name__') else schema}")
    return f"MockLLM({model})"


# ตัวอย่างการใช้งานใน nodes
print("=== LLM Factory Usage ===")
print()

print("1. User Goal Extraction (structured output):")
llm_goal = make_llm(model="gpt-4o-mini", schema=UserGoalOut)
print(f"   Result: {llm_goal}")
print()

print("2. Route Classification (structured output):")
llm_route = make_llm(model="gpt-4o-mini", schema=RouteOut)
print(f"   Result: {llm_route}")
print()

print("3. Answer Builder (raw output):")
llm_raw = make_llm(model="gpt-4.1-mini")  # ใช้ model ที่ดีกว่าสำหรับ generation
print(f"   Result: {llm_raw}")
print()

print("หมายเหตุ: @lru_cache ทำให้ call ซ้ำไม่สร้าง instance ใหม่")
llm_again = make_llm(model="gpt-4o-mini", schema=UserGoalOut)
print(f"   ครั้งที่ 2 (cached): {llm_again is llm_goal}")

## Task Lifecycle — วงจรชีวิตของ Task

In [ ]:
import uuid
from datetime import datetime

# Task Lifecycle ใน state
# โค้ดจริง: src/agent/utils/task_state_stub.py

def init_new_task(state: dict) -> dict:
    """เริ่ม task ใหม่"""
    task_id = str(uuid.uuid4())[:8]
    return {
        "active_task_id": task_id,
        "task_status": "ACTIVE",
        "task_first_user_message": state.get("last_user_message", ""),
    }


def finalize_task(state: dict, answer: str) -> dict:
    """
    จบ task:
    1. สร้าง task summary
    2. บันทึกลง task_summaries
    3. Reset task state
    """
    task_summary = {
        "task_id": state.get("active_task_id"),
        "first_user_message": state.get("task_first_user_message", ""),
        "user_goal": state.get("user_goal", ""),
        "outcome": answer[:100],  # เก็บแค่ 100 chars
        "timestamp": datetime.now().isoformat(),
    }
    
    summaries = list(state.get("task_summaries", []))
    summaries.append(task_summary)
    summaries = summaries[-10:]  # เก็บแค่ 10 รายการล่าสุด
    
    return {
        "task_status": "COMPLETED",
        "final_answer": answer,
        "task_summaries": summaries,
        "last_task_summary": task_summary,
        # Reset task-specific fields
        "user_goal": "",
        "topic_intents": [],
        "hospital_results": [],
        "ready_to_answer": False,
    }


# Simulate task lifecycle
state = {
    "last_user_message": "ที่จอดรถโรงพยาบาล A มีกี่ที่",
    "task_status": "COMPLETED",
    "task_summaries": [],
}

print("=== Task Lifecycle Simulation ===")
print(f"Initial: task_status={state['task_status']}")
print()

# Start new task
new_task = init_new_task(state)
state.update(new_task)
print(f"After init: task_id={state['active_task_id']}, status={state['task_status']}")

# Add user_goal
state["user_goal"] = "ต้องการทราบข้อมูลที่จอดรถของโรงพยาบาล A"
print(f"user_goal: {state['user_goal']!r}")

# Finalize task
final = finalize_task(state, "โรงพยาบาล A มีที่จอดรถ 500 คัน ชั้น B1-B3")
state.update(final)
print(f"\nAfter finalize:")
print(f"  task_status: {state['task_status']}")
print(f"  final_answer: {state['final_answer']!r}")
print(f"  summaries: {len(state['task_summaries'])} items")

## สรุป Graph Architecture

```
Main Graph (graph_stub.py)
├── session_gate        → เตรียม context
│     └── router → intent_classify / resume_router / handoff_mode
│
├── intent_classify     → LLM วิเคราะห์ intent → route_module
│     └── router → hospital_info / nurse / package / concierge
│
├── resume_router       → ต่อ task เดิม
│     └── router → ตาม current_agent_module
│
├── handoff_mode        → จัดการ HANDOFF state
│
└── Module Nodes (subgraphs)
    ├── hospital_info  → extract → fetch CSV → build answer
    ├── nurse          → (similar structure)
    ├── package        → (similar structure)
    └── concierge      → เมนูหลัก + route ไปยัง module อื่น
```

**Notebook ถัดไป:** `05_session_memory.ipynb` — ดูระบบ session management